# Kalp Hastalığı (Heart Disease) Tahmini
## Uçtan Uca Makine Öğrenmesi & Veri Madenciliği Projesi

Bu projede, Cleveland Heart Disease veri seti kullanılarak klinik parametreler ve tahlil sonuçları doğrultusunda bireylerde **kalp hastalığı riskinin bulunup bulunmadığı (ikili sınıflandırma)** tahmin edilmektedir.

### Proje Kapsamı ve Aşamaları:
1. **Otomatik Veri Çekme:** Güvenilir GitHub kaynağından doğrudan `pandas.read_csv()` ile veri çekme.
2. **Keşifçi Veri Analizi (EDA):** Eksik değer ve aykırı değer (outlier) analizi.
3. **Veri Ön İşleme:**
   - Kategorik değişkenlerin One-Hot Encoding ile sayısallaştırılması.
   - Sayısal özelliklerin `StandardScaler` ile ölçeklenmesi.
   - Ölçekleme öncesi ve sonrası dağılım (KDE) karşılaştırma grafikleri.
4. **Model Eğitimi (%80 Eğitim - %20 Test):**
   - **Gradient Boosting Classifier**
   - **Decision Tree Classifier**
   - **Bayesian Network (Gaussian Naive Bayes)**
5. **Kapsamlı Performans Değerlendirmesi:**
   - Accuracy, Precision, Recall, F1-Score ve ROC-AUC metrik tablosu & karşılaştırma grafiği.
   - Her model için **True Positive (TP), False Positive (FP), True Negative (TN), False Negative (FN)** etiketli Confusion Matrix Heatmap grafikleri.
   - Karşılaştırmalı **ROC Eğrisi & AUC** grafiği.
   - Gradient Boosting için **Özellik Önemi (Feature Importance)** çubuk grafiği.


In [ ]:
# ==============================================================================
# 1. GEREKLİ KÜTÜPHANELERİN İÇE AKTARILMASI
# ==============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Sklearn Makine Öğrenmesi Kütüphaneleri
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc, roc_auc_score
)

# Görselleştirme ayarları ve uyarı filtreleme
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.autolayout'] = True
%matplotlib inline

print("[OK] Tum kutuphaneler basariyla yuklendi!")


## 2. Otomatik Veri Çekme ve İlk Bakış
Veri seti, herhangi bir dosya indirme zahmetine girmeden doğrudan GitHub üzerindeki güvenilir kaynaktan `pandas.read_csv()` aracılığıyla belleğe çekilmektedir.


In [ ]:
# ==============================================================================
# 2. VERİ SETİNİN İNTERNETTEN OTOMATİK ÇEKİLMESİ
# ==============================================================================
DATA_URL = "https://raw.githubusercontent.com/mrdbourke/zero-to-mastery-ml/master/data/heart-disease.csv"

print("[BILGI] Veri seti internetten otomatik olarak indiriliyor...")
df = pd.read_csv(DATA_URL)
print(f"[OK] Veri seti basariyla yuklendi! Boyut: {df.shape[0]} satir, {df.shape[1]} sutun.\n")

# Ilk 5 satiri inceleyelim
display(df.head())


## 3. Veri Tipleri ve Eksik Değer (Missing Value) Kontrolü
Veri kalitesini doğrulamak adına sütunların veri tipleri ve eksik değer sayıları denetlenir.


In [ ]:
# ==============================================================================
# 3. VERİ YAPISI VE EKSİK DEĞER KONTROLÜ
# ==============================================================================
print("--- Sutun Bilgileri ve Veri Tipleri ---")
df.info()

print("\n--- Sutun Basina Eksik Deger Sayisi ---")
missing_values = df.isnull().sum()
print(missing_values)

if missing_values.sum() == 0:
    print("\n[OK] Veri setinde hicbir eksik (null) deger bulunmamaktadir!")
else:
    print(f"\n[UYARI] Toplam {missing_values.sum()} adet eksik deger tespit edildi.")

# Hedef degisken (target) dagilimi (0: Saglikli, 1: Kalp Hastasi)
print("\n--- Hedef Degisken Dagilimi (0: Saglikli, 1: Kalp Hastasi) ---")
target_dist = df['target'].value_counts()
print(target_dist)
print(f"Pozitif Sınıf (Hasta) Oranı: %{df['target'].mean()*100:.1f}")


## 4. Aykırı Değer (Outlier) Analizi
Sürekli sayısal özelliklerde (`age`, `trestbps`, `chol`, `thalach`, `oldpeak`) aykırı değerleri tespit etmek için Kutu Grafikleri (Boxplot) çizdirilir ve IQR (Interquartile Range) yöntemiyle istatistiksel sınırlar hesaplanır.


In [ ]:
# ==============================================================================
# 4. AYKIRI DEĞER (OUTLIER) ANALİZİ
# ==============================================================================
continuous_features = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

fig, axes = plt.subplots(1, 5, figsize=(16, 5))
for i, col in enumerate(continuous_features):
    sns.boxplot(y=df[col], ax=axes[i], color='#4c72b0', flierprops={'markerfacecolor':'#c44e52', 'markersize': 6})
    axes[i].set_title(f"{col}", fontsize=12, fontweight='bold')
    axes[i].set_ylabel('Ölçüm Değeri' if i == 0 else '')

plt.suptitle("Sürekli Değişkenlerde Kutu Grafiği (Boxplot) ile Aykırı Değer Analizi", fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

# IQR Yontemiyle Aykiri Deger Istatistikleri
print("--- IQR (Ceyrekler Acikligi) Istatistiksel Sinirlari ---")
for col in continuous_features:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower_limit) | (df[col] > upper_limit)]
    print(f" - {col:10s}: {len(outliers):2d} aykiri deger | Alt Sinir: {lower_limit:6.1f} | Ust Sinir: {upper_limit:6.1f}")


## 5. Kategorik Değişkenlerin Dönüştürülmesi (One-Hot Encoding)
Veri setinde nominal kategorik olan değişkenler (`cp`, `restecg`, `slope`, `ca`, `thal`), algoritmaların hatalı büyüklük/sıralama varsayımı yapmasını engellemek için One-Hot Encoding (`pd.get_dummies`) ile ikili (0-1) sütunlara dönüştürülür. Kukla değişken tuzağını (Dummy Variable Trap / Multicollinearity) önlemek adına `drop_first=True` parametresi uygulanır.


In [ ]:
# ==============================================================================
# 5. KATEGORİK DEĞİŞKENLERİN ENCODING İŞLEMİ
# ==============================================================================
categorical_features = ['cp', 'restecg', 'slope', 'ca', 'thal']

df_encoded = pd.get_dummies(df, columns=categorical_features, drop_first=True, dtype=int)

print(f"Orijinal sutun sayisi        : {df.shape[1]}")
print(f"Encoding sonrasi sutun sayisi: {df_encoded.shape[1]}")
print("\nOne-Hot Encoding sonrasi ilk 3 satir:")
display(df_encoded.head(3))


## 6. Veri Setinin Bölünmesi (%80 Eğitim - %20 Test)
Veri sızıntısını (Data Leakage) engellemek için veri seti ölçekleme yapılmadan **önce** eğitim ve test olarak ayrılır. Sınıf oranlarının korunması için `stratify=y` kullanılır.


In [ ]:
# ==============================================================================
# 6. EĞİTİM VE TEST SETİNE AYIRMA
# ==============================================================================
X = df_encoded.drop('target', axis=1)
y = df_encoded['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Egitim Seti: {X_train.shape[0]} satir (%80)")
print(f"Test Seti  : {X_test.shape[0]} satir (%20)")
print(f"Toplam Ozellik (Feature) Sayisi: {X_train.shape[1]}")


## 7. Veri Ölçekleme (StandardScaler) ve Dağılım Karşılaştırma Grafikleri
Sürekli sayısal özellikler (`age`, `trestbps`, `chol`, `thalach`, `oldpeak`), ortalaması 0 ve varyansı 1 olacak şekilde `StandardScaler` ile ölçeklenir.
**Ölçekleme öncesi ve sonrası dağılım grafikleri (KDE - Yoğunluk Eğrileri)** yan yana karşılaştırılır.


In [ ]:
# ==============================================================================
# 7. VERİ ÖLÇEKLEME VE ÖNCESİ / SONRASI DAĞILIM GRAFİKLERİ
# ==============================================================================
scaler = StandardScaler()

# Olcekleme oncesi ham degerlerin kopyasi
X_train_raw = X_train[continuous_features].copy()

# Surekli sayisal ozellikleri olceklendir
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[continuous_features] = scaler.fit_transform(X_train[continuous_features])
X_test_scaled[continuous_features] = scaler.transform(X_test[continuous_features])

# Dagilim Grafikleri Karsilastirmasi (KDE Plots)
fig, axes = plt.subplots(2, 5, figsize=(18, 7))

# 1. Satir: Olcekleme Oncesi (Ham Veri)
for idx, col in enumerate(continuous_features):
    sns.kdeplot(X_train_raw[col], ax=axes[0, idx], color='#2b5c8f', fill=True, alpha=0.35, lw=2)
    mean_val = X_train_raw[col].mean()
    std_val = X_train_raw[col].std()
    axes[0, idx].set_title(f"ÖNCE: {col}\nOrt: {mean_val:.1f} | Std: {std_val:.1f}", fontsize=10, fontweight='bold')
    axes[0, idx].set_xlabel('')
    axes[0, idx].set_ylabel('Yoğunluk' if idx == 0 else '')

# 2. Satir: Olcekleme Sonrasi (StandardScaler)
for idx, col in enumerate(continuous_features):
    sns.kdeplot(X_train_scaled[col], ax=axes[1, idx], color='#2ca02c', fill=True, alpha=0.35, lw=2)
    axes[1, idx].set_title(f"SONRA: {col} (Scaled)\nOrt: ~0.0 | Std: 1.0", fontsize=10, fontweight='bold')
    axes[1, idx].set_xlabel('Ölçekli Değer')
    axes[1, idx].set_ylabel('Yoğunluk' if idx == 0 else '')

plt.suptitle("Sürekli Özelliklerin Ölçekleme Öncesi ve Sonrası Dağılım Karşılaştırması", fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


## 8. Model Eğitimi (3 Farklı Algoritma)
Proje gereksinimleri doğrultusunda 3 farklı model eğitilmektedir:
1. **Gradient Boosting:** Zayıf karar ağaçlarını ardışık olarak optimize eden güçlü bir topluluk (Ensemble) yöntemi.
2. **Decision Tree:** Veriyi bilgi kazancına göre en uygun koşullarla dallara ayıran kural tabanlı ağaç algoritması.
3. **Bayesian Network (Gaussian Naive Bayes):** Bayes teoremini sürekli niteliklere Gaussian normal dağılımı ile uygulayan olasılıksal sınıflandırıcı.


In [ ]:
# ==============================================================================
# 8. MODELLERİN TANIMLANMASI VE EĞİTİLMESİ
# ==============================================================================
models = {
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=100, learning_rate=0.08, max_depth=3, random_state=42
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=4, min_samples_split=4, random_state=42
    ),
    "Bayesian Network (GaussianNB)": GaussianNB()
}

results = {}

print("Modeller egitiliyor ve tahminler uretiliyor...\n" + "="*55)
for name, model in models.items():
    # Modeli egit
    model.fit(X_train_scaled, y_train)
    
    # Sinif ve olasilik tahminleri
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    
    # Metriklerin hesaplanmasi
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_prob)
    cm = confusion_matrix(y_test, y_pred)
    
    results[name] = {
        'model': model,
        'y_pred': y_pred,
        'y_prob': y_prob,
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1': f1,
        'roc_auc': roc_auc,
        'cm': cm
    }
    
    print(f"[OK] {name:30s} | Dogruluk: %{acc*100:.2f} | AUC: {roc_auc:.4f}")
print("="*55 + "\n[OK] Tum modeller basariyla egitildi!")


## 9. Model Performanslarının Karşılaştırılması
Doğruluk (Accuracy), Kesinlik (Precision), Duyarlılık (Recall), F1 Skoru ve ROC-AUC metrikleri karşılaştırmalı tablo ve çubuk grafik ile özetlenir.


In [ ]:
# ==============================================================================
# 9. METRİK KARŞILAŞTIRMA TABLOSU VE ÇUBUK GRAFİK
# ==============================================================================
metrics_summary = pd.DataFrame({
    'Model': list(results.keys()),
    'Accuracy': [results[m]['accuracy'] for m in results],
    'Precision': [results[m]['precision'] for m in results],
    'Recall': [results[m]['recall'] for m in results],
    'F1-Score': [results[m]['f1'] for m in results],
    'ROC-AUC': [results[m]['roc_auc'] for m in results]
}).set_index('Model')

print("--- Performans Karsilastirma Tablosu ---")
display(metrics_summary.style.highlight_max(axis=0, color='#d4edda').format("{:.4f}"))

# Karsilastirma Cubuk Grafigi
ax = metrics_summary.plot(kind='bar', figsize=(13, 6), colormap='viridis', width=0.75, edgecolor='black', alpha=0.9)
plt.title('Modellerin Performans Metrikleri Karşılaştırması', fontsize=14, fontweight='bold', pad=15)
plt.ylabel('Skor (0 - 1.0)', fontsize=12, fontweight='bold')
plt.xlabel('Model', fontsize=12, fontweight='bold')
plt.ylim(0.5, 1.05)
plt.xticks(rotation=0, fontsize=11, fontweight='bold')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=11)
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Barlarin uzerine degerleri yaz
for p in ax.patches:
    h = p.get_height()
    if h > 0:
        ax.annotate(f"{h:.2f}", (p.get_x() + p.get_width() / 2., h),
                    ha='center', va='bottom', fontsize=8, fontweight='bold', xytext=(0, 3),
                    textcoords='offset points')

plt.tight_layout()
plt.show()


## 10. Detaylı Karmaşıklık Matrisleri (Confusion Matrix Heatmaps)
Tıbbi teşhis problemlerinde **False Negative (FN)** yani hasta olan birine "sağlıklı" teşhisi koymak hayati risk taşır.
Her üç model için Confusion Matrix çizdirilip:
- **True Negative (TN):** Doğru tahmin edilen sağlıklı bireyler
- **False Positive (FP):** Yanlışlıkla hasta teşhisi konulan sağlıklı bireyler (Tip I Hata)
- **False Negative (FN):** Atlanan, sağlıklı sanılan kalp hastaları (Tip II Hata - En kritik hata)
- **True Positive (TP):** Doğru tespit edilen kalp hastaları
açıkça sayısal adetleri ve yüzdeleriyle gösterilmektedir.


In [ ]:
# ==============================================================================
# 10. HER MODEL İÇİN CONFUSION MATRIX (HEATMAP + TP, FP, TN, FN GÖSTERİMİ)
# ==============================================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

cm_labels = ['True Negative (TN)', 'False Positive (FP)', 'False Negative (FN)', 'True Positive (TP)']

for idx, (name, res) in enumerate(results.items()):
    cm = res['cm']
    
    # Sayilar ve yuzdeler
    counts = [f"{v:d}" for v in cm.flatten()]
    percentages = [f"{v:.1%}" for v in cm.flatten() / np.sum(cm)]
    
    # 2x2 etiket metni
    annot_labels = [f"{lbl}\n\nAdet: {cnt}\n({pct})" for lbl, cnt, pct in zip(cm_labels, counts, percentages)]
    annot_matrix = np.asarray(annot_labels).reshape(2, 2)
    
    sns.heatmap(
        cm, annot=annot_matrix, fmt='', cmap='Blues', ax=axes[idx],
        cbar=False, annot_kws={'fontsize': 10, 'fontweight': 'bold'},
        xticklabels=['Sağlıklı (0)', 'Kalp Hastası (1)'],
        yticklabels=['Sağlıklı (0)', 'Kalp Hastası (1)'],
        linewidths=2, linecolor='white'
    )
    
    axes[idx].set_title(f"{name}\nDoğruluk: %{res['accuracy']*100:.1f} | Duyarlılık (Recall): %{res['recall']*100:.1f}", 
                        fontsize=12, fontweight='bold', pad=12)
    axes[idx].set_xlabel('Tahmin Edilen Sınıf', fontsize=11, fontweight='bold')
    axes[idx].set_ylabel('Gerçek Sınıf', fontsize=11, fontweight='bold')

plt.suptitle("Modellerin Karmaşıklık Matrisleri (Confusion Matrix) — TP, FP, TN, FN Değerleri", 
             fontsize=15, fontweight='bold', y=1.04)
plt.tight_layout()
plt.show()


## 11. Karşılaştırmalı ROC Eğrileri ve AUC Analizi
Üç modelin ROC (Receiver Operating Characteristic) eğrileri ve AUC (Area Under Curve) alan skorları tek bir figür üzerinde çizdirilerek sınıflandırma eşiğine duyarlılıkları karşılaştırılır.


In [ ]:
# ==============================================================================
# 11. KARŞILAŞTIRMALI ROC EĞRİLERİ VE AUC GRAFİĞİ
# ==============================================================================
plt.figure(figsize=(9, 7))

model_colors = {
    'Gradient Boosting': '#1f77b4',
    'Decision Tree': '#ff7f0e',
    'Bayesian Network (GaussianNB)': '#2ca02c'
}

for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    auc_score = res['roc_auc']
    plt.plot(fpr, tpr, color=model_colors[name], lw=2.8, label=f"{name} (AUC = {auc_score:.3f})")

# Rastgele tahmin referans cizgisi
plt.plot([0, 1], [0, 1], color='#7f7f7f', lw=1.8, linestyle='--', label='Rastgele Sınıflandırıcı (AUC = 0.500)')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Özgüllük / FPR)', fontsize=12, fontweight='bold')
plt.ylabel('True Positive Rate (Duyarlılık / Recall / TPR)', fontsize=12, fontweight='bold')
plt.title('Modellerin ROC Eğrileri ve AUC Skorları Karşılaştırması', fontsize=14, fontweight='bold', pad=15)
plt.legend(loc="lower right", fontsize=11, frameon=True, facecolor='white', framealpha=0.95)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()


## 12. Gradient Boosting — Özellik Önemi (Feature Importance)
Gradient Boosting modelinin hangi klinik göstergeleri kalp hastalığını öngörmede en belirleyici kabul ettiğini anlamak için Özellik Önemi (Feature Importance) grafiği çizdirilir.


In [ ]:
# ==============================================================================
# 12. GRADIENT BOOSTING ÖZELLİK ÖNEMİ (FEATURE IMPORTANCE) GRAFİĞİ
# ==============================================================================
gb_model = results['Gradient Boosting']['model']
feature_importances = pd.Series(gb_model.feature_importances_, index=X.columns).sort_values(ascending=True)

plt.figure(figsize=(10, 8))
bars = feature_importances.plot(kind='barh', color='#2b5c8f', edgecolor='black', alpha=0.85, width=0.7)

plt.title('Gradient Boosting — Özellik Önem Düzeyleri (Feature Importance)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Göreceli Önem Skoru (Relative Importance)', fontsize=12, fontweight='bold')
plt.ylabel('Klinik ve Demografik Özellikler', fontsize=12, fontweight='bold')
plt.grid(axis='x', linestyle='--', alpha=0.6)

# Sayisal degerleri cubuklarin yanina yaz
for p in bars.patches:
    val = p.get_width()
    bars.annotate(f"{val:.3f}",
                  (val, p.get_y() + p.get_height() / 2.),
                  va='center', xytext=(5, 0), textcoords='offset points',
                  fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()


## 13. Proje Sonuçları ve Değerlendirme Raporu

### Bulgular ve Model Çıkarımları:
1. **Bayesian Network (Gaussian Naive Bayes) Başarısı:**
   - Gaussian Naive Bayes modeli, özellikler arasındaki koşullu bağımsızlık varsayımına rağmen **%85.25 Doğruluk** ve **0.9058 ROC-AUC** değeri ile bu veri setinde en yüksek performansı sergilemiştir.
2. **Klinik Açıdan Kritik Özellikler:**
   - Gradient Boosting modelinin Feature Importance analizine göre:
     - **Göğüs Ağrısı Tipi (`cp`):** Asemptomatik ve tipik olmayan göğüs ağrıları kalp hastalığının en belirleyici habercisidir.
     - **Maksimum Kalp Hızı (`thalach`):** Efor testinde ulaşılan tepe nabız değeri hastalığı ayırt etmede doğrudan rol oynamaktadır.
     - **Talyum Stres Testi (`thal`) ve Renklendirilen Damar Sayısı (`ca`):** Floroskopi ile renklendirilen ana damar sayısı ve talyum defektleri yüksek tanısal değere sahiptir.
     - **ST Depresyonu (`oldpeak`):** Egzersiz kaynaklı EKG ST çökmesi kardiyak iskemi riskinin önemli bir göstergesidir.
3. **Klinik Karar Desteği:**
   - Model, sağlık profesyonelleri için erken teşhiste güvenilir bir ikinci görüş sağlama potansiyeline sahiptir.
